In [28]:
import numpy as np
import pandas as pd

df = pd.read_csv('car_fuel_efficiency_2026.csv')
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
df = df[base + ['fuel_efficiency_mpg']]

def split_data(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    return (df.iloc[idx[:n_train]],
            df.iloc[idx[n_train:n_train + n_val]],
            df.iloc[idx[n_train + n_val:]])

def train_linear_regression_reg(X, y, r=0.0):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T.dot(X) + r * np.eye(X.shape[1])
    w = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

def prepare_X(df, fill):
    return df[base].fillna(fill).values

In [29]:
# Missing values
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [30]:
# Median for horsepower
df.horsepower.median()

np.float64(254.0)

In [31]:
# Filling NAs
df_train, df_val, df_test = split_data(df, 42)
y_train, y_val = df_train.fuel_efficiency_mpg.values, df_val.fuel_efficiency_mpg.values

for name, fill in [('0', 0), ('mean', df_train.horsepower.mean())]:
    w0, w = train_linear_regression_reg(prepare_X(df_train, fill), y_train)
    print(name, round(rmse(y_val, w0 + prepare_X(df_val, fill).dot(w)), 3))

0 2.205
mean 2.202


In [32]:
# Best regularization
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(prepare_X(df_train, 0), y_train, r)
    print(r, round(rmse(y_val, w0 + prepare_X(df_val, 0).dot(w)), 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [33]:
# RMSE standard deviation
scores = []
for seed in range(10):
    tr, va, _ = split_data(df, seed)
    w0, w = train_linear_regression_reg(prepare_X(tr, 0), tr.fuel_efficiency_mpg.values)
    scores.append(rmse(va.fuel_efficiency_mpg.values, w0 + prepare_X(va, 0).dot(w)))

round(np.std(scores), 3)

np.float64(0.029)

In [34]:
# Evaluation on test
df_train, df_val, df_test = split_data(df, 9)
df_full = pd.concat([df_train, df_val])

w0, w = train_linear_regression_reg(prepare_X(df_full, 0), df_full.fuel_efficiency_mpg.values, r=0.001)
round(rmse(df_test.fuel_efficiency_mpg.values, w0 + prepare_X(df_test, 0).dot(w)), 3)

np.float64(2.236)